# Day 2 · 01 · Power BI Semantic Model on Databricks

![Power BI report mockup](../../assets/images/powerbi_report_mockup.png)

Day 1 produced a trusted Gold model. Day 2 starts with the Power BI semantic model: which Databricks tables Power BI should use, which logic belongs in Power BI, and how refresh mode affects cost and performance.


## Business Scenario

RetailHub wants to publish an executive Sales dashboard and a drill-through detail page. The Gold model is ready, but the team still needs a semantic model decision: star schema or wide table, Import or DirectQuery, and how incremental refresh should filter data.


## Learning Objectives

By the end of this notebook you can:

- define a Power BI semantic model and explain how it relates to Databricks Gold,
- choose between star schema and wide table consumption,
- explain Import, DirectQuery and Composite modes,
- simulate an incremental-refresh date window,
- recognize query folding and Power BI Performance Analyzer signals,
- prepare a BI contract before the hands-on Workshop 2 build.


## Setup

Resolve the participant catalog. `00_setup` executes `USE CATALOG`, so SQL cells can use two-part names such as `gold.fact_sales_dashboard`.


In [ ]:
%run ../../setup/00_setup


### Configuration

Confirm the active catalog and schemas before running the Day 2 examples.


In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))


### Runtime Pre-check

Day 2 starts from the Gold model created in Workshop 1. This check fails early if the model is not available.


In [ ]:
required_tables = [
    f"{GOLD}.dim_date",
    f"{GOLD}.dim_customer",
    f"{GOLD}.dim_product",
    f"{GOLD}.fact_sales",
    f"{GOLD}.fact_sales_dashboard",
]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    for table in missing:
        print("[MISSING]", table)
    raise Exception("Pre-check failed. Run Workshop 1 solution before starting Day 2.")
print("[OK] Day 1 Gold model is available.")


## 1. Semantic Model Definition

A **Power BI semantic model** is the governed layer that contains tables, relationships, measures, formatting and refresh behavior. Reports query the semantic model; the semantic model queries Databricks.

Databricks Gold should provide trusted rows and reusable fields. Power BI should provide report-specific relationships, measures and user interactions.

Expected observation: semantic modeling is a contract decision, not only a UI step in Power BI Desktop.


In [ ]:
%sql
SELECT 'dim_date' AS object_name, COUNT(*) AS rows FROM gold.dim_date
UNION ALL
SELECT 'dim_customer', COUNT(*) FROM gold.dim_customer
UNION ALL
SELECT 'dim_product', COUNT(*) FROM gold.dim_product
UNION ALL
SELECT 'fact_sales', COUNT(*) FROM gold.fact_sales
UNION ALL
SELECT 'fact_sales_dashboard', COUNT(*) FROM gold.fact_sales_dashboard
ORDER BY object_name


## 2. Star Schema vs Wide Table Decision

A **star schema** uses one fact table plus dimensions. It is best for governed semantic models because relationships and measures are reusable.

A **wide table** repeats dimension attributes on the fact rows. It is useful for quick prototypes or narrow dashboard pages, but it can scan more data in DirectQuery.


In [ ]:
%sql
SELECT
  'star_fact' AS source_option,
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(DISTINCT order_id) AS distinct_orders
FROM gold.fact_sales
UNION ALL
SELECT
  'wide_dashboard',
  COUNT(*),
  COUNT(DISTINCT line_id),
  COUNT(DISTINCT order_id)
FROM gold.fact_sales_dashboard


Expected observation: both options should keep line grain. The decision is not about correctness only; it is about semantic-model reuse and query cost.


## 3. Relationship Readiness

**Relationship readiness** means dimension keys are unique and fact rows can match those keys. Power BI many-to-one relationships depend on this condition.

An **orphan check** finds fact rows without matching dimensions. Orphans usually become blanks in slicers and can confuse users.


In [ ]:
%sql
SELECT 'dim_date' AS dimension, COUNT(*) AS rows, COUNT(DISTINCT date_key) AS distinct_keys, COUNT(*) - COUNT(DISTINCT date_key) AS duplicate_keys
FROM gold.dim_date
UNION ALL
SELECT 'dim_customer', COUNT(*), COUNT(DISTINCT customer_id), COUNT(*) - COUNT(DISTINCT customer_id)
FROM gold.dim_customer
UNION ALL
SELECT 'dim_product', COUNT(*), COUNT(DISTINCT product_id), COUNT(*) - COUNT(DISTINCT product_id)
FROM gold.dim_product


In [ ]:
%sql
SELECT 'fact_sales.order_date -> dim_date.date_key' AS relationship, COUNT(*) AS orphan_rows
FROM gold.fact_sales f
LEFT ANTI JOIN gold.dim_date d ON f.order_date = d.date_key
UNION ALL
SELECT 'fact_sales.customer_id -> dim_customer.customer_id', COUNT(*)
FROM gold.fact_sales f
LEFT ANTI JOIN gold.dim_customer c ON f.customer_id = c.customer_id
UNION ALL
SELECT 'fact_sales.product_id -> dim_product.product_id', COUNT(*)
FROM gold.fact_sales f
LEFT ANTI JOIN gold.dim_product p ON f.product_id = p.product_id


## 4. Import, DirectQuery and Composite

![Import vs DirectQuery](../../assets/images/import_vs_directquery.png)

`Import` copies data into Power BI memory during refresh. It is usually fastest for dashboards and safest for cost.

`DirectQuery` sends each interaction back to the SQL Warehouse. It is useful for live requirements, but it can multiply queries through visuals and slicers.

`Composite` mixes Import and DirectQuery tables. It solves mixed freshness requirements, but it increases model complexity.


In [ ]:
%sql
WITH scenarios AS (
  SELECT 'Executive dashboard' AS report_area, 'Import' AS recommended_mode, 'fast interactions and scheduled refresh' AS reason
  UNION ALL SELECT 'Operational live monitor', 'DirectQuery', 'freshness requirement is stronger than latency risk'
  UNION ALL SELECT 'Hybrid management report', 'Composite', 'import dimensions and aggregates, query live detail only when needed'
)
SELECT *
FROM scenarios


## 5. Incremental Refresh Definition

**Incremental refresh** reloads only selected date partitions instead of the full dataset. Power BI commonly uses parameters named `RangeStart` and `RangeEnd`.

Use a half-open interval: `order_datetime >= RangeStart AND order_datetime < RangeEnd`. This avoids double-counting records exactly at midnight between partitions.


In [ ]:
%sql
WITH params AS (
  SELECT
    TIMESTAMP '2024-01-01 00:00:00' AS RangeStart,
    TIMESTAMP '2024-02-01 00:00:00' AS RangeEnd
)
SELECT
  p.RangeStart,
  p.RangeEnd,
  COUNT(*) AS rows_in_window,
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS completed_revenue
FROM gold.fact_sales_dashboard f
CROSS JOIN params p
WHERE CAST(f.order_date AS TIMESTAMP) >= p.RangeStart
  AND CAST(f.order_date AS TIMESTAMP) < p.RangeEnd
GROUP BY p.RangeStart, p.RangeEnd


## 6. Query Folding Definition

**Query folding** means Power Query can translate transformations into source SQL. When folding works, Databricks does the filtering and aggregation. When folding breaks, Power BI may generate inefficient work.

Professional rule: do heavy shaping in Databricks Gold and keep Power Query transformations simple, especially for DirectQuery.


In [ ]:
%sql
EXPLAIN
SELECT
  year_month,
  channel,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue
FROM gold.fact_sales_dashboard
WHERE order_date >= DATE '2024-01-01'
  AND order_date < DATE '2024-02-01'
GROUP BY year_month, channel


## 7. Connector Walkthrough

![Power BI connection walkthrough](../../assets/images/powerbi_connection_walkthrough.png)

Power BI needs SQL Warehouse connection details, not a notebook URL.

Connection checklist:

| Field | Source |
|---|---|
| Server hostname | Databricks SQL Warehouse connection details |
| HTTP path | Databricks SQL Warehouse connection details |
| Authentication | workspace policy: Entra ID, OAuth or token |
| Catalog/schema | participant catalog, `gold` schema |
| Recommended first table | `gold.fact_sales_dashboard` for quick prototype; star schema for governed model |


## 8. Performance Analyzer Awareness

Power BI Performance Analyzer shows which visuals trigger slow DAX queries or source queries. For Databricks DirectQuery, pair it with SQL Warehouse Query History.

Expected observation: a slow report page is investigated from both sides: Power BI visual fan-out and Databricks SQL execution profile.


## 9. BI Contract Before Workshop 2

Workshop 2 will build the dataset/performance layer. The contract is:

| Object | Built in | Purpose |
|---|---|---|
| `gold.fact_sales_dashboard_monthly` | W2 | import-friendly monthly aggregate |
| `gold.v_fact_sales_incremental` | W2 | detail view with `order_datetime` for incremental refresh |
| `gold.kpi_monthly` | W2 | KPI-card source and sanity-check table |

Expected observation: Day2_01 explains the decisions. Workshop 2 creates the durable serving objects.


## Summary and Workshop Handoff

You now have the semantic model vocabulary and the contract for Workshop 2: build smaller, refresh-ready, performance-aware objects for Power BI without changing the Day 1 Gold model.
